# 06 — Power BI Gold Export and Handoff

**Week:** 8

## Purpose

This notebook prepares the approved Week 7 Gold outputs for Power BI and documents the Gold handoff used for dashboard development.

## Week 8 Boundary

This notebook will:

- verify the approved Gold handoff from Week 7
- inventory the Gold tables available in the project environment
- validate selected Gold table grain, keys, columns, and measures
- export only the Gold tables selected for Power BI
- read back exported data for validation
- support Power BI handoff and reconciliation
- preserve evidence of the Gold outputs used during Week 8

Power BI will use approved Gold outputs only.

No raw, Bronze, Candidate Silver, Quarantine, or unapproved Silver tables will be used as Power BI sources.

## Gold Tables Selected for Power BI Dashboard

The following four validated Gold tables are the direct Power BI dashboard sources for Week 8:

1. `gold_monthly_collision_trends`
2. `gold_vehicle_type_readable`
3. `gold_casualty_severity_readable`
4. `gold_vehicle_propulsion_readable`

Other validated Gold tables may be retained for Gold-layer information, validation, and handoff proof but are not direct Power BI dashboard sources.

## Week 8 Rule

The Gold tables remain the owning source for report measures.

Power BI should aggregate approved Gold measures and should not recreate Silver-to-Gold transformations.


## 1. Gold Source Register

The register below documents the four Gold tables selected for the Power BI dashboard. The complete Gold inventory is generated from the actual Databricks catalog in the next cell so that no table names are fabricated in the notebook.

| Gold table | Grain | KPI / Business Purpose | Power BI Dashboard |
|---|---|---|---|
| `gold_monthly_collision_trends` | Collision year + collision month + urban/rural group | Monthly collision volume and severe-collision metrics | Yes |
| `gold_vehicle_type_readable` | Collision year + vehicle type | Vehicle involvement by vehicle type | Yes |
| `gold_casualty_severity_readable` | Collision year + casualty age group + casualty severity | Casualty counts by severity and age group | Yes |
| `gold_vehicle_propulsion_readable` | Collision year + propulsion code | Vehicle counts by propulsion code | Yes |

The remaining validated Gold tables are retained as Gold-layer outputs and are not connected directly to the Week 8 dashboard.


In [ ]:
# 2. Inventory actual Gold tables
# This reads the Databricks catalog so the notebook records real Gold tables only.

all_tables = spark.catalog.listTables()

gold_tables = sorted([
    t.name
    for t in all_tables
    if t.name.startswith("gold_") and t.tableType != "TEMPORARY"
])

print("Gold tables found in the current Databricks environment:")
for table_name in gold_tables:
    print("-", table_name)

print("\nTotal Gold tables found:", len(gold_tables))


In [ ]:
# 3. Verify the four dashboard Gold tables exist

dashboard_gold_tables = [
    "gold_monthly_collision_trends",
    "gold_vehicle_type_readable",
    "gold_casualty_severity_readable",
    "gold_vehicle_propulsion_readable"
]

missing_dashboard_tables = [
    table_name
    for table_name in dashboard_gold_tables
    if table_name not in gold_tables
]

if missing_dashboard_tables:
    raise ValueError(
        "Missing approved dashboard Gold tables: "
        + ", ".join(missing_dashboard_tables)
    )

print("All four approved dashboard Gold tables are present.")
for table_name in dashboard_gold_tables:
    print("Verified:", table_name)


## 4. Gold Table Profiles

Each selected Gold table is profiled independently before export. The profile records row count and column names without changing the Gold data.


In [ ]:
# 5. Profile the four Gold tables selected for Power BI

for table_name in dashboard_gold_tables:
    df = spark.table(table_name)
    row_count = df.count()
    print("\nTable:", table_name)
    print("Row count:", row_count)
    print("Columns:")
    for field in df.schema.fields:
        print("  -", field.name, ":", field.dataType.simpleString())


## 6. Selected Gold Table Validation

The selected Gold tables are checked individually before export. The checks focus on the approved report-facing structure and the presence of expected business fields.


In [ ]:
# 7. Validate expected report-facing columns

expected_columns = {
    "gold_monthly_collision_trends": [
        "collision_year",
        "collision_month",
        "month_name",
        "month_sort",
        "period_label",
        "urban_rural_group",
        "collision_count",
        "severe_collision_count",
        "severe_collision_rate"
    ],
    "gold_vehicle_type_readable": [
        "collision_year",
        "vehicle_type_name",
        "vehicle_involvement_count"
    ],
    "gold_casualty_severity_readable": [
        "collision_year",
        "casualty_age_group",
        "casualty_severity",
        "casualty_severity_name",
        "casualty_count"
    ],
    "gold_vehicle_propulsion_readable": [
        "collision_year",
        "propulsion_code",
        "propulsion_type_name",
        "vehicle_count"
    ]
}

for table_name, required_columns in expected_columns.items():
    actual_columns = spark.table(table_name).columns
        missing = [c for c in required_columns if c not in actual_columns]

    if missing:
        raise ValueError(
            f"{table_name} is missing expected columns: {missing}"
        )

    print("Validated columns:", table_name)


In [ ]:
# 8. Check report-facing measure validity

measure_checks = {
    "gold_monthly_collision_trends": [
        "collision_count",
        "severe_collision_count",
        "severe_collision_rate"
    ],
    "gold_vehicle_type_readable": [
        "vehicle_involvement_count"
    ],
    "gold_casualty_severity_readable": [
        "casualty_count"
    ],
    "gold_vehicle_propulsion_readable": [
        "vehicle_count"
    ]
}

for table_name, measures in measure_checks.items():
    df = spark.table(table_name)

    for measure in measures:
        null_count = df.filter(df[measure].isNull()).count()
        negative_count = df.filter(df[measure] < 0).count()
        
        if null_count > 0:
            raise ValueError(
                f"{table_name}.{measure} contains {null_count} null values"
            )

        if negative_count > 0:
            raise ValueError(
                f"{table_name}.{measure} contains {negative_count} negative values"
            )

    print("Validated measures:", table_name)


## 9. Export Contract

Only the four selected dashboard Gold tables are exported for Power BI.

Export rules:

- Export directly from approved Gold tables.
- Do not edit business values during export.
- Preserve the approved Gold grain.
- Preserve report-facing columns.
- Use deterministic ordering where practical.
- Keep each Gold table as a separate export.
- Do not combine different-grain Gold tables into one export.


In [ ]:
# 10. Create controlled Gold exports
# Adjust this base path only if the approved Databricks Volume path differs.

export_base = "/Volumes/workspace/default/gold_exports"

export_order = {
    "gold_monthly_collision_trends": [
        "collision_year",
        "collision_month",
        "month_name",
        "month_sort",
        "period_label",
        "urban_rural_group",
        "collision_count",
        "severe_collision_count",
        "severe_collision_rate"
    ],
    "gold_vehicle_type_readable": [
        "collision_year",
        "vehicle_type_name",
        "vehicle_involvement_count"
    ],
    "gold_casualty_severity_readable": [
        "collision_year",
        "casualty_age_group",
        "casualty_severity",
        "casualty_severity_name",
        "casualty_count"
    ],
    "gold_vehicle_propulsion_readable": [
        "collision_year",
        "propulsion_code",
        "propulsion_type_name",
        "vehicle_count"
    ]
}

for table_name in dashboard_gold_tables:
    df = spark.table(table_name).select(*export_order[table_name])

    sort_columns = export_order[table_name]
    df = df.orderBy(*sort_columns)

    output_path = f"{export_base}/{table_name}"

    df.coalesce(1).write \
        .mode("overwrite") \
        .option("header", True) \
        .csv(output_path)

    print("Exported:", table_name)
    print("Path:", output_path)


## 11. Export Read-Back Proof

The exported files are read back to confirm that the Power BI handoff contains the expected report-facing columns and row counts.


In [ ]:
# 12. Read back each export and compare row counts

for table_name in dashboard_gold_tables:
    source_df = spark.table(table_name).select(*export_order[table_name])
    exported_df = spark.read.option("header", True).option("inferSchema", True).csv(
        f"{export_base}/{table_name}"
    )

    source_count = source_df.count()
    exported_count = exported_df.count()
        
        print("\nTable:", table_name)
    print("Gold row count:", source_count)
    print("Export row count:", exported_count)

    if source_count != exported_count:
        raise ValueError(
            f"Row-count mismatch for {table_name}: "
            f"Gold={source_count}, Export={exported_count}"
        )


In [ ]:
# 13. Reconcile selected Gold measures to the exported data

for table_name in dashboard_gold_tables:
    source_df = spark.table(table_name)
    exported_df = spark.read.option("header", True).option("inferSchema", True).csv(
        f"{export_base}/{table_name}"
    )

    if table_name == "gold_monthly_collision_trends":
        source_values = source_df.agg(
            {
                "collision_count": "sum",
                "severe_collision_count": "sum"
            }
        ).collect()[0]
        export_values = exported_df.agg(
            {
                "collision_count": "sum",
                "severe_collision_count": "sum"
            }
        ).collect()[0]
    elif table_name == "gold_vehicle_type_readable":
        source_values = source_df.agg({"vehicle_involvement_count": "sum"}).collect()[0]
        export_values = exported_df.agg({"vehicle_involvement_count": "sum"}).collect()[0]
    elif table_name == "gold_casualty_severity_readable":
        source_values = source_df.agg({"casualty_count": "sum"}).collect()[0]
        export_values = exported_df.agg({"casualty_count": "sum"}).collect()[0]
    else:
        source_values = source_df.agg({"vehicle_count": "sum"}).collect()[0]
        export_values = exported_df.agg({"vehicle_count": "sum"}).collect()[0]

    print("Reconciled export measures:", table_name)
    print("Gold:", source_values)
    print("Export:", export_values)


## 14. Power BI Handoff

The four exported Gold tables are the approved direct dashboard sources for Week 8.

Power BI modeling must keep the four Gold tables separate unless a relationship is explicitly justified by a shared key and compatible grain.

Different-grain Gold summary tables must not be directly joined merely because they contain a similarly named column.

Report measures must trace back to their owning Gold table.


## 15. Evidence to Capture

Capture genuine execution evidence for:

- Gold source register
- Gold export and reconciliation
- Power BI model
- Dashboard pages
- Measure reconciliation

Evidence should show actual Databricks or Power BI execution and should not contain fabricated values, screenshots, PASS results, or table histories.


## 16. Week 8 Close

The Week 8 handoff is complete when the selected Gold tables have been validated, exported or connected through an approved Gold-only path, loaded into Power BI, reconciled to their owning Gold measures, and supported by genuine execution evidence.

Week 9 refinement and insight storytelling are outside the scope of this notebook.
